# Constant polynomial addition

Adapted from frozen Qualtran v0.7.0 at bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2025 Google LLC, Apache-2.0. All original example instances, parameters, register topics, signatures and graph nodes are retained. Native names and table layout replace the original presentation.


In [ ]:
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.field_polynomial_catalog import polynomial_addition_examples, polynomial_example_graph
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg


## Operation, parameters and ownership

The input f_x is a THRU polynomial register. The constant g_x must have degree at most the declared degree and belong to the same nominal coefficient field. For concrete execution, pass descending encoded coefficients. Leading zeros are padded through the existing layout. The output is |f_x + g_x>. The original symbolic example keeps a zero polynomial over GF(2), skips concrete field checks and declares an unknown coefficient _g_x0 with the width upper bound.

The native field-polynomial codec owns numerical values. Compiler owns packed coefficient views and gate emission. Coefficients run from the highest power to the constant, with big-endian bits inside each word. All physical wires belong to the caller. No work wires or measurements are allocated.


In [ ]:
from flagquantum.compiler.registers.finite_field.field_polynomial_addition import gf2_polynomial_add_ir, gf2_polynomial_add_constant_ir, source_gf2_polynomial_add_resource_formula, source_gf2_polynomial_add_constant_resource_formula
from flagquantum.compiler.registers.finite_field.field_polynomial_layout import FieldPolynomialLayout
from examples.algorithms.binary_field_arithmetic import FieldOperation
import flagquantum as fq
import torch
from collections import Counter


### Example Instances


In [ ]:
examples = polynomial_addition_examples(constant=True)
concrete = examples[0]
concrete


In [ ]:
symbolic = examples[1]
symbolic


#### Graphical Signature


In [ ]:
display(SVG(field_signature_svg([(name, signature, {}) for name, signature, _ in examples])))


### Call Graph

The concrete constant graph includes polynomial split and join leaves, just as the original graph does. These view nodes emit zero gates. Source declared counts and emitted primitives are separate accounts.


In [ ]:
graph = polynomial_example_graph(concrete, max_depth=1)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
print("One-level declared leaves", graph.leaf_counts)
print("Expanded declared leaves", polynomial_example_graph(concrete, max_depth=None).leaf_counts)


## Execute through append

This small executable instance uses degree one and two-bit coefficients. Every complex amplitude is checked, with a complex spectator and a clean unused wire. The operation is self-inverse. There is one coherent branch. Physical primitive counts and the source formula are printed separately.

The source-specific adjoint reverses coefficient calls. Its register XOR children retain their canonical forward bit order. The generic circuit inverse is also numerically exact and reverses every primitive.


In [ ]:
layout = FieldPolynomialLayout(1, 2)
wires = (3, 1, 4, 2)
n = 6
state = torch.arange(1, 33, dtype=torch.float64) + 1j * torch.arange(32, dtype=torch.float64)
state = state / state.norm()
state = torch.stack((state, torch.zeros_like(state)), dim=1).flatten()
instructions = gf2_polynomial_add_constant_ir(wires, (1, 3), layout)
mask = (1 << (n - 1 - wires[1])) | (1 << (n - 1 - wires[2])) | (1 << (n - 1 - wires[3]))
expected = state[torch.arange(64) ^ mask]
model = source_gf2_polynomial_add_constant_resource_formula(1, 2, coefficients=(1, 3))
operation = FieldOperation(n, instructions)
circuit = fq.Circuit(n, dtype=torch.complex128, inputs=state).append(operation)
torch.testing.assert_close(circuit.state().flatten(), expected, rtol=0, atol=1e-12)
assert torch.count_nonzero(circuit.state().flatten()[1::2]) == 0
inverse_gates = gf2_polynomial_add_constant_ir(wires, (1, 3), layout, adjoint=True)
torch.testing.assert_close(circuit.append(FieldOperation(n, inverse_gates)).state().flatten(), state, rtol=0, atol=1e-12)
print("Actual emitted primitives", dict(Counter(g.name for g in instructions)))
print("Source model", model)
print("Auxiliary wires and measurements", 0, 0)
